# CIC-IDS2017: внесок кожної пастки в +0.264 (старі таймстемпи, однаковий бюджет)

Це відсутній рядок для таблиці «конфігурація оцінювання → ΔAP».

1. **Санітарна перевірка.** Відтворюємо результат статті на таймстемпах як є (без виправлення AM/PM), з early stopping: F0 0.710369 @4, F1 0.974028 @118.
2. **Той самий спліт, однаковий бюджет.** F0 і F1 порівнюються при 4, 11, 50, 118, 163, 300 і 500 деревах. Так видно, яку частину +0.264 дає саме різниця в кількості дерев, а яку дає сам спліт.
3. **При 500 деревах** рахуємо блочний bootstrap CI для F1−F0 і для F0+port−F0.

In [1]:
import pandas as pd, numpy as np, lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

PATH = "/srv/ids_research/data/enriched/cic_ids2017_with_cve_context.parquet"
TS_COL, LABEL_COL, PORT_COL = "Timestamp", "attack_label", "destination_port"
SEED, ROUNDS, N_BOOT = 42, 500, 500

F0_CIC = ['Flow Duration','Total Fwd Packets','Total Backward Packets','Total Length of Fwd Packets',
'Total Length of Bwd Packets','Fwd Packet Length Max','Fwd Packet Length Min','Fwd Packet Length Mean',
'Fwd Packet Length Std','Bwd Packet Length Max','Bwd Packet Length Min','Bwd Packet Length Mean',
'Bwd Packet Length Std','Flow Bytes/s','Flow Packets/s','Flow IAT Mean','Flow IAT Std','Flow IAT Max',
'Flow IAT Min','Fwd IAT Total','Fwd IAT Mean','Fwd IAT Std','Fwd IAT Max','Fwd IAT Min','Bwd IAT Total',
'Bwd IAT Mean','Bwd IAT Std','Bwd IAT Max','Bwd IAT Min','Fwd PSH Flags','Bwd PSH Flags','Fwd URG Flags',
'Bwd URG Flags','Fwd Header Length','Bwd Header Length','Fwd Packets/s','Bwd Packets/s','Min Packet Length',
'Max Packet Length','Packet Length Mean','Packet Length Std','Packet Length Variance','FIN Flag Count',
'SYN Flag Count','RST Flag Count','PSH Flag Count','ACK Flag Count','URG Flag Count','CWE Flag Count',
'ECE Flag Count','Down/Up Ratio','Average Packet Size','Avg Fwd Segment Size','Avg Bwd Segment Size',
'Fwd Avg Bytes/Bulk','Fwd Avg Packets/Bulk','Fwd Avg Bulk Rate','Bwd Avg Bytes/Bulk','Bwd Avg Packets/Bulk',
'Bwd Avg Bulk Rate','Subflow Fwd Packets','Subflow Fwd Bytes','Subflow Bwd Packets','Subflow Bwd Bytes',
'Init_Win_bytes_forward','Init_Win_bytes_backward','act_data_pkt_fwd','min_seg_size_forward','Active Mean',
'Active Std','Active Max','Active Min','Idle Mean','Idle Std','Idle Max','Idle Min','protocol']
CVE_CTX = ['cve_ctx_service','cve_ctx_pool_size','cve_ctx_n_epss','cve_ctx_epss_coverage',
           'cve_ctx_service_max_epss','cve_ctx_n_generic_os_matches','cve_ctx_n_specific_product_matches']
F1_CIC = F0_CIC + CVE_CTX
CAT_COLS = {"protocol", "cve_ctx_service"}

df = pd.read_parquet(PATH)
df.columns = [c.strip() for c in df.columns]
y_np = (df[LABEL_COL].astype(str) != "BENIGN").astype(int).to_numpy()

# Таймстемпи ЯК Є (без виправлення AM/PM), так само, як у пайплайні статті
raw = df[TS_COL]
ts = pd.to_datetime(raw) if np.issubdtype(raw.dtype, np.datetime64) else pd.to_datetime(raw.astype(str), dayfirst=True, errors="coerce")
b1 = ts.quantile(0.70, interpolation="nearest"); b2 = ts.quantile(0.85, interpolation="nearest")
tr = (ts < b1).to_numpy(); va = ((ts >= b1) & (ts < b2)).to_numpy(); te = (ts >= b2).to_numpy()
print("межі:", b1, "|", b2, "  (у статті: 2017-07-06 09:28:00 | 2017-07-07 03:23:00)")
print("розміри:", int(tr.sum()), int(va.sum()), int(te.sum()), "  (у статті: 1,979,346 / 419,191 / 429,140)")
print("AP випадкового класифікатора на test:", round(float(y_np[te].mean()), 6))

межі: 2017-07-06 09:28:00 | 2017-07-07 03:23:00   (у статті: 2017-07-06 09:28:00 | 2017-07-07 03:23:00)
розміри: 1979346 419191 429140   (у статті: 1,979,346 / 419,191 / 429,140)
AP випадкового класифікатора на test: 0.31222


In [2]:
SPW = (y_np[tr] == 0).sum() / y_np[tr].sum()
PARAMS = dict(objective="binary", metric="None", seed=SEED, num_leaves=15, learning_rate=0.05,
              min_data_in_leaf=50, verbosity=-1, scale_pos_weight=SPW)
print("scale_pos_weight:", round(float(SPW), 4), "  (у статті: 6.4454)")

def prep(cols):
    frame = df[cols].copy()
    cats = [c for c in CAT_COLS if c in cols]
    for c in cats:
        frame[c] = pd.Categorical(frame[c], categories=pd.Categorical(frame.loc[tr, c]).categories)
    return frame, cats

def ap_feval(preds, data):
    return "unweighted_ap", average_precision_score(data.get_label(), preds), True

# 1. Санітарна перевірка: відтворення статті (early stopping на val)
for k, cols, target in [("F0", F0_CIC, "0.710369 @4"), ("F1", F1_CIC, "0.974028 @118")]:
    frame, cats = prep(cols)
    dtr = lgb.Dataset(frame[tr], label=y_np[tr], categorical_feature=cats, free_raw_data=False)
    dva = lgb.Dataset(frame[va], label=y_np[va], categorical_feature=cats, reference=dtr, free_raw_data=False)
    m = lgb.train(PARAMS, dtr, num_boost_round=ROUNDS, valid_sets=[dva], feval=ap_feval,
                  callbacks=[lgb.early_stopping(30, verbose=False)])
    pr = m.predict(frame[te], num_iteration=m.best_iteration)
    print(f"[{k} early-stop] best_iter={m.best_iteration} test_AP={average_precision_score(y_np[te], pr):.6f}   (у статті: {target})")

scale_pos_weight: 6.4454   (у статті: 6.4454)
[F0 early-stop] best_iter=4 test_AP=0.710369   (у статті: 0.710369 @4)
[F1 early-stop] best_iter=118 test_AP=0.974028   (у статті: 0.974028 @118)


In [3]:
# 2. Той самий (невиправлений) спліт, однаковий бюджет
models = {}
for k, cols in [("F0", F0_CIC), ("F1", F1_CIC), ("F0+port", F0_CIC + [PORT_COL])]:
    frame, cats = prep(cols)
    models[k] = (lgb.train(PARAMS, lgb.Dataset(frame[tr], label=y_np[tr], categorical_feature=cats),
                           num_boost_round=ROUNDS), frame)
    print(f"[{k}] навчено {ROUNDS} дерев")

rows = []
for r in [4, 11, 50, 118, 163, 300, 500]:
    a = {k: average_precision_score(y_np[te], m.predict(f[te], num_iteration=r)) for k, (m, f) in models.items()}
    rows.append({"rounds": r, **{f"AP_{k}": v for k, v in a.items()}, "dAP_F1_F0": a["F1"] - a["F0"]})
    print(f"rounds={r:>3}  F0={a['F0']:.6f}  F1={a['F1']:.6f}  dAP={a['F1'] - a['F0']:+.6f}  F0+port={a['F0+port']:.6f}")
curve = pd.DataFrame(rows)

[F0] навчено 500 дерев
[F1] навчено 500 дерев
[F0+port] навчено 500 дерев
rounds=  4  F0=0.710369  F1=0.893411  dAP=+0.183042  F0+port=0.749120
rounds= 11  F0=0.679626  F1=0.897459  dAP=+0.217833  F0+port=0.750543
rounds= 50  F0=0.894842  F1=0.957343  dAP=+0.062501  F0+port=0.751001
rounds=118  F0=0.903585  F1=0.974028  dAP=+0.070442  F0+port=0.955815
rounds=163  F0=0.905089  F1=0.974087  dAP=+0.068998  F0+port=0.933334
rounds=300  F0=0.903481  F1=0.966695  dAP=+0.063214  F0+port=0.929268
rounds=500  F0=0.910831  F1=0.962910  dAP=+0.052078  F0+port=0.943431


In [5]:
# 3. Блочний bootstrap CI при 500 деревах (блоки = хвилини)
p = {k: m.predict(f[te], num_iteration=ROUNDS) for k, (m, f) in models.items()}
y_te = y_np[te]
blk = pd.factorize(ts[te].dt.floor("min"))[0]
blocks = list(pd.Series(np.arange(len(blk))).groupby(blk).indices.values())
rng = np.random.default_rng(SEED)

def block_ci(pa, pb):
    point = average_precision_score(y_te, pa) - average_precision_score(y_te, pb)
    vals = []
    for _ in range(N_BOOT):
        j = np.concatenate([blocks[k] for k in rng.integers(0, len(blocks), len(blocks))])
        vals.append(average_precision_score(y_te[j], pa[j]) - average_precision_score(y_te[j], pb[j]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return point, lo, hi, float(np.mean(np.array(vals) > 0))

res = []
for name, a, b in [("F1 − F0", "F1", "F0"), ("F0+port − F0", "F0+port", "F0"), ("F1 − F0+port", "F1", "F0+port")]:
    pt, lo, hi, share = block_ci(p[a], p[b])
    res.append({"comparison": name, "dAP": pt, "CI95_lo": lo, "CI95_hi": hi, "P(dAP>0)": share})
    print(f"{name:<14} dAP={pt:+.6f}  95% block-CI [{lo:+.6f}, {hi:+.6f}]  частка реплік >0: {share:.3f}")

out = "/srv/ids_research/data"
curve.to_csv(f"{out}/cic_asparsed_samebudget_curve.csv", index=False)
pd.DataFrame(res).to_csv(f"{out}/cic_asparsed_samebudget_ci.csv", index=False)
print("Збережено: cic_asparsed_samebudget_curve.csv, cic_asparsed_samebudget_ci.csv")

F1 − F0        dAP=+0.052078  95% block-CI [+0.041952, +0.066013]  частка реплік >0: 1.000
F0+port − F0   dAP=+0.032600  95% block-CI [+0.017369, +0.044178]  частка реплік >0: 1.000
F1 − F0+port   dAP=+0.019478  95% block-CI [+0.015136, +0.026173]  частка реплік >0: 1.000
Збережено: cic_asparsed_samebudget_curve.csv, cic_asparsed_samebudget_ci.csv
